# <p style="font-family:Arial, sans-serif; font-size:180%; text-align:center; color:#1A365D;">🏦 Silicon Valley Bank (SVB) Collapse</p>
## <p style="text-align:center; color:#2B6CB0;">End-to-End Web Scraping, 10 NLP Preprocessing Steps, and Vector Embedding Pipeline</p>
---
### 📌 Project Overview
This notebook implements an automated, production-ready NLP workflow covering:
1. **Live Web Scraping** from international reputable news and regulatory sources (*Reuters, BBC News, Federal Reserve Reports, Wikipedia*).
2. **Comprehensive NLP Preprocessing (10 Sequential Steps)**: Lowercasing, HTML stripping, URL cleaning, Punctuation removal, Financial/Chat Acronyms expansion, Character normalization, Stopword filtering, Emoji handling, Stemming, and Lemmatization.
3. **Tokenization & Vocabulary Indexing**: Token-to-ID mapping and subword readiness.
4. **Semantic Sliding-Window Chunking**: Structuring text into chunks optimized for Retrieval-Augmented Generation (RAG) and Vector Stores.
5. **Dense Vector Embeddings & Similarity Retrieval**: Computing vector matrices and demonstrating instant semantic query search on the SVB corpus.

In [ ]:
# Step 0: Imports & Environment Configuration
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pipeline Modules
from scraper import SVBScraper
from preprocessor import SVBTextPreprocessor, preprocess_articles_dataset
from vector_embedding_prep import SVBVectorEmbeddingPreparer, export_vector_dataset

# Display Settings
pd.set_option('display.max_colwidth', 200)
print("✅ All dependencies and pipeline modules imported successfully!")

## 🌐 1. Scraping Reputable International Sources on SVB Collapse
We scrape multi-source journalistic and regulatory reports on the March 2023 collapse of Silicon Valley Bank.

In [ ]:
# Run Multi-Source Scraper
raw_articles = SVBScraper.scrape_all_sources(force_live=True)
print(f"Scraped {len(raw_articles)} source documents.")

# Inspect Scraped Sources
source_summary = [
    {
        "Publisher": a["source_publisher"],
        "Title": a["title"],
        "Paragraph Count": len(a["paragraphs"]),
        "URL": a["source_url"]
    }
    for a in raw_articles
]
pd.DataFrame(source_summary)

## 🧹 2. Executing 10 NLP Preprocessing Steps
We now apply the sequential 10-step cleaning pipeline:

In [ ]:
# Preprocess all scraped articles
processed_records = preprocess_articles_dataset(raw_articles)
print(f"Total Preprocessed Paragraphs: {len(processed_records)}")

# Create DataFrame for inspection
df_preprocessed = pd.DataFrame(processed_records)
df_preprocessed[["record_id", "publisher", "raw_text", "cleaned_text", "token_count"]].head(5)

### 🔍 Step-by-Step Intermediate NLP Transformation Inspector
Let's view the exact before-and-after transformation for a single paragraph across all intermediate stages.

In [ ]:
sample = processed_records[0]
print("=" * 80)
print("ORIGINAL RAW TEXT:")
print(sample["raw_text"])
print("-" * 80)
print("INTERMEDIATE TRANSFORMATIONS:")
for step_name, value in sample["intermediate_steps"].items():
    print(f"  • {step_name}: {value[:120]}...")
print("-" * 80)
print("FINAL CLEANED & NORMALIZED TEXT:")
print(sample["cleaned_text"])
print("=" * 80)

## 🧩 3. Tokenization & Semantic Sliding-Window Chunking
For Vector Databases and RAG, chunking text into overlapping windows (e.g. 180 tokens with 30-token overlap) ensures context is preserved across chunk boundaries.

In [ ]:
preparer = SVBVectorEmbeddingPreparer(chunk_word_size=180, chunk_overlap=30)
chunks = preparer.create_semantic_chunks(processed_records)
print(f"Generated {len(chunks)} Semantic Chunks.")
print(f"Vocabulary Size: {len(preparer.vocab_to_id)} distinct tokens.")

# Display Sample Chunks
df_chunks = pd.DataFrame(chunks)
df_chunks[["chunk_id", "publisher", "source_title", "token_count", "cleaned_text"]].head(5)

## ⚡ 4. Dense Vector Embeddings Generation
We compute dense vector embeddings with normalized feature matrices ready for Vector DB ingestion.

In [ ]:
embeddings = preparer.fit_and_compute_embeddings(chunks)
print(f"Embedding Matrix Shape: {embeddings.shape[0]} chunks x {embeddings.shape[1]} vector dimensions.")

# Sample Vector Representation for Chunk 1
sample_chunk = chunks[0]
print(f"Chunk ID: {sample_chunk['chunk_id']}")
print(f"Embedding Vector Length: {len(sample_chunk['embedding_vector'])}")
print(f"Non-zero features: {sample_chunk['non_zero_features']}")
print(f"Vector Slice (First 10 non-zero dims): {[v for v in sample_chunk['embedding_vector'] if v > 0][:10]}")

## 🔎 5. Interactive Semantic Search / Vector Retrieval Demo
Test natural language queries against the SVB Collapse vector space.

In [ ]:
query = "Why did Federal Reserve interest rate hikes cause bond portfolio losses for Silicon Valley Bank?"
results = preparer.search_similar_chunks(query, top_k=3)

print(f"🔍 QUERY: {query}\n")
for r in results:
    print(f"⭐ Rank #{r['rank']} | Cosine Similarity Score: {r['similarity_score']:.4f} | ID: {r['chunk_id']}")
    print(f"   Source: [{r['publisher']}] {r['source_title']}")
    print(f"   Excerpt: {r['cleaned_text'][:250]}...\n")

## 💾 6. Exporting All Datasets
We export the structured vector embeddings to JSON (ready for ChromaDB/Pinecone/FAISS) and CSV.

In [ ]:
export_vector_dataset(
    chunks=chunks,
    vocab_to_id=preparer.vocab_to_id,
    json_path="svb_collapse_vector_embeddings.json",
    csv_path="svb_chunks_dataset.csv"
)
print("✨ All datasets exported successfully!")
print("  • svb_collapse_raw_scraped.json")
print("  • svb_collapse_preprocessed.json")
print("  • svb_collapse_vector_embeddings.json")
print("  • svb_chunks_dataset.csv")